# 分析报告

## 数据说明
### 个股选择
- 本文选择股票如下：   
  
| 代码 | 名称 | 行业 | 选股理由 |
| :--- | :--- | :--- | :--- |
| 600036 | 招商银行 | 银行 | A股零售银行业务最突出的股份制银行，不良率长期优于同业，2020年成为首家跻身万亿市值俱乐部的股份行，市场认可度高。 |
| 600104 | 上汽集团 | 汽车 | A股规模最大的整车上市公司，1997年上市，产业链覆盖完整，自主品牌与合资品牌均衡布局，行业龙头地位稳固。 |
| 000002 | 万科A | 房地产 | 中国房地产行业最早的上市龙头之一，连续多年位居住宅开发行业第一，品牌力与经营稳健性行业领先。 |
| 600048 | 保利地产 | 房地产 | 央企背景的大型国有房地产企业，股权分置改革后首批上市的地产股，土地储备充裕，销售规模稳居行业前五。 |
| 600519 | 贵州茅台 | 白酒 | A股价值投资的标杆，2020年市值超越工商银行登顶A股首位，凭借极强的品牌护城河和定价权，长期保持业绩高增长。 |
| 000858 | 五粮液 | 白酒 | 浓香型白酒龙头，2020年市值突破万亿，跻身A股市值前十，品牌力与渠道能力仅次于茅台，成长确定性高。 |
| 601857 | 中国石油 | 能源 | 国内最大的油气生产和销售商，全产业链布局完善，2020年虽受国际油价下行冲击，但资源储备和上游产能优势仍为行业核心。 |
| 601898 | 中煤能源 | 能源 | 国内第二大煤炭企业，集煤炭生产、煤化工、煤矿装备制造于一体，一体化经营模式增强抗风险能力，在煤炭行业处于领先地位。 |
| 600050 | 中国联通 | 通讯 | 三大通信运营商中最早登陆A股的标的，2002年即上市，凭借5G建设和数字化转型，2020年在行业竞争中保持稳健市场份额。 |
| 600233 | 圆通速递 | 物流 | 中国快递行业首家登陆A股的公司，自有航空运力构建差异化竞争优势，2020年在快递行业竞争加剧背景下仍保持利润正增长。 |
   
- 下载每只股票过去 5 年（2020-01-01 至今）的后复权日度行情，字段包含：日期、开盘价、收盘价、最高价、最低价、成交量、成交额
### 指数
- 选择沪深 300（000300）和创业板指（399006）作为后续 CAPM 分析的市场基准，因创业板指（399006）与沪深300风格差异显著、β值高，便于清晰验证CAPM中“高风险对应高收益”的核心逻辑。
- 同样下载指数过去 5 年（2020-01-01 至今）的后复权日度行情，字段包含：日期、开盘价、收盘价、最高价、最低价、成交量、成交额
### 宏观经济指标   
- 选择CPI 同比增速和M2 同比增速，M2同比增速是衡量宏观流动性的核心指标，而流动性是影响股市走势的最直接因素之一。下载宏观经济指标过去 5 年（2020-01-01 至今）的月度数据。
### 财务指标
- 获取 10 只股票最近 5 个年度的净资产收益率（ROE）和资产负债率，将财务数据整理为长格式（Long format）：每行为一只股票一个年度的观测，字段包含 code, year, indicator, value。   
    


##  清洗说明   
在此阶段完成以下功能：   
### 单表清洗（对每只股票的原始数据）   

| 清洗项目 | 具体要求 |
| :--- | :--- |
| 缺失值检测 | 统计每列缺失值的数量和比例，制成表格；说明缺失的可能原因（停牌？节假日？数据源问题？） |
| 缺失值处理 | 向前填充（ffill）或删除，须说明选择依据 |
| 日期格式统一 | 确保所有表的日期列统一为 datetime64 格式，并设为索引 |
| 数据类型检查 | 确认价格、成交量列为数值型；若存在字符型需转换并记录 |
| 重复值处理 | 检测并删除重复行，记录删除数量 |
| 离群值标注 | 计算日收益率，对单日涨跌幅超过 ±20% 的记录，在新列 is_extreme 中标注为 True（不删除，但须说明可能成因） |   

- 结果发现：不存在缺失值，重复值和离群值，遂未进行处理
### 宽表与长表转换
- 将 10 只股票的收盘价合并为宽表（日期为索引，每列一只股票），再用 pd.melt 转换回长表，字段为 date, code, close。
- 宽表（每行日期、每列股票）适合横截面分析、相关性矩阵计算、投资组合优化、滚动窗口运算和一次性可视化多只股票对比；长表（每行是日期-股票组合）适合按股票分组操作（如计算单只股票的累计收益率）、与指数/宏观数据合并、数据库存储、以及基于股票代码的筛选和聚合（如计算每只股票的平均成交量）。
### 多表合并
- 将个股日度数据与指数日度数据按日期做 left join，将月度宏观数据与日度数据合并（将宏观数据映射到对应月份的每个交易日）
- 代码运行会记录每次合并前后的行数，并输出行数变化的原因
### 两种格式对比：  
- CSV    读取耗时: 0.040s  文件大小: 967.9 KB
- Parquet读取耗时: 0.008s  文件大小: 169.4 KB
- 速度提升: 4.76x
- 空间节省: 82.5%
- 在本次数据规模（约1MB级别）下，Parquet相比CSV速度提升近5倍、体积减少82.5%，差异已经非常显著；而在更大数据量（如千万行以上）、只读取部分列、网络传输或频繁读写等场景下，Parquet的列式存储、高效压缩和类型保留特性会使其优势进一步扩大到10-50倍，差异将更为悬殊。 
     


## 统计结果
### 基本统计量
10 只股票日收益率的描述性统计如下:
| 股票 | 行业 | 年化均值 | 年化波动率 | 偏度 | 峰度 | 最大回撤 |
| :--- | :--- | ---: | ---: | ---: | ---: | ---: |
| 招商银行 | 银行 | 0.050409 | 0.277503 | 0.262583 | 3.144393 | -0.509439 |
| 上汽集团 | 汽车 | -0.051758 | 0.317016 | 0.337340 | 5.294789 | -0.539865 |
| 万科A | 房地产 | -0.315200 | 0.364080 | 0.654080 | 3.257207 | -0.861036 |
| 保利地产 | 房地产 | -0.126860 | 0.361261 | 0.561637 | 3.235698 | -0.657729 |
| 贵州茅台 | 白酒 | 0.066582 | 0.277202 | 0.261575 | 3.605799 | -0.474821 |
| 五粮液 | 白酒 | -0.011530 | 0.345692 | 0.086992 | 3.307914 | -0.660939 |
| 中国石油 | 能源 | 0.169138 | 0.292717 | 0.220927 | 5.178174 | -0.324707 |
| 中煤能源 | 能源 | 0.241632 | 0.392553 | 0.281857 | 2.679710 | -0.387922 |
| 中国联通 | 通讯 | -0.023345 | 0.290338 | 0.900426 | 7.498519 | -0.435733 |
| 圆通速递 | 物流 | 0.093493 | 0.368407 | 0.681860 | 2.727214 | -0.511971 |   

## 图表
**1.归一化收盘价走势图**
- 10 只股票以 2020-01-01 = 1 为基准的归一化收盘价，叠加沪深 300。   
    
![归一化收盘价走势图](./output/normalized_price.png)   
    
- 招商银行、贵州茅台等股票在2020年之后整体表现相对稳健，而中国石油、中煤能源等周期股价格波动较大呈现上升趋势，而房地产行业的股票相较于其他行业股票呈现出下降趋势。   
     
**2.日收益率分布图**
- 10 只股票收益率分面直方图（2 行 × 5 列），每个子图叠加正态分布曲线   

![日收益率分布图](./output/return_distribution.png)    
    
- 招商银行的均值收益率为0.0002，标准差为0.0175，说明其收益波动较小，表现相对稳定；而中煤能源标准差高达0.0247，波动性明显更大。五粮液和圆通速递的均值接近零，但标准差分别为0.0218和0.0232，显示其收益波动性较高，适合风险承受能力较强的投资者。    
     
**3.收益率相关系数热力图**
- 10 只股票日收益率的相关系数矩阵热力图，标注具体数值   
   
![收益率相关系数热力图](./output/correlation_heatmap.png)    
    
- 同行业内部平均相关性：
- 白酒: 0.8346，房地产: 0.8205，能源: 0.5388，跨行业平均相关性: 0.2939
- 结论：同行业内部相关性普遍高于跨行业平均相关性，表明行业因素对股价走势有显著影响。   
     

**4.宏观指标与股市关系**
- 绘制CPI同比增速与沪深 300 月度收益率的散点图，叠加线性拟合线   
   
![宏观指标与股市关系](./output/cpi_vs_hs300.png)   
    
- Pearson相关系数: -0.0777, p值: 0.5104   
- 经济含义讨论：CPI与股市收益率呈负相关关系。   
- 可能原因：
1. 高通胀预期可能导致央行收紧货币政策，对股市形成压力
2. 通胀上升侵蚀企业利润，降低股票估值
3. 投资者可能转向抗通胀资产，减少股票配置
## CAPM 结果
### CAPM 模型估计
| 股票 | 行业 | α | α_p值 | β | R² |
| :--- | :--- | ---: | ---: | ---: | ---: |
| 招商银行 | 银行 | 0.000148 | 0.675806 | 0.905174 | 0.378071 |
| 上汽集团 | 汽车 | -0.000254 | 0.567079 | 0.849890 | 0.255394 |
| 万科A | 房地产 | -0.001308 | 0.009652 | 1.001204 | 0.268719 |
| 保利地产 | 房地产 | -0.000552 | 0.293957 | 0.846064 | 0.194899 |
| 贵州茅台 | 白酒 | 0.000208 | 0.534647 | 0.978052 | 0.442360 |
| 五粮液 | 白酒 | -0.000120 | 0.762986 | 1.295298 | 0.498891 |
| 中国石油 | 能源 | 0.000644 | 0.154446 | 0.469905 | 0.091574 |
| 中煤能源 | 能源 | 0.000926 | 0.130450 | 0.567663 | 0.074307 |
| 中国联通 | 通讯 | -0.000133 | 0.749095 | 0.714330 | 0.215098 |
| 圆通速递 | 物流 | 0.000324 | 0.549846 | 0.821237 | 0.176574 |
   
### 可视化
- 绘制 Beta 系数点图，横轴为 Beta 值，纵轴为股票名称，误差棒表示 95% 置信区间，按行业分组着色，在β=1处画参考竖线。   
   
![Beta 系数点图](./output/beta_dot_plot.png)    
   
### 分析讨论
1. β > 1的股票：
   万科A (房地产): β = 1.0012
   五粮液 (白酒): β = 1.2953

    与'周期性 vs 防御性'行业分类的吻合性分析：
  - β > 1表示股票波动性大于市场，属于'进攻型'或'周期性'行业
  - β < 1表示股票波动性小于市场，属于'防御型'行业
  - 从结果来看：   
    - ✓ 万科A(房地产)符合周期性行业特征，房地产政策往往在一段时间内呈单边趋势（如持续从严或持续宽松），这种政策惯性容易催生板块的趋势性行情，为进攻提供宏观环境。
    - ? 五粮液(白酒)通常被视为兼具进攻与防御属性的“双栖型”行业，白酒是典型的顺周期消费品，其需求与商务活动、居民收入预期高度相关。当宏观经济向好时，白酒板块往往能获得超越大盘的涨幅。白酒承载文化和情感属性，能够获取高附加值，龙头品牌拥有深厚的“护城河”，在行业下行期具备抗跌能力。

2. α显著异于零的股票（p值 < 0.05）：
  万科A (房地产): α = -0.001308 (负值), p值 = 0.0097

  - α显著的经济含义：股票表现低于风险调整后的预期，可能存在管理问题或负面因素


3. R²最高的股票：五粮液 (白酒), R² = 0.4989   
   R²最低的股票：中煤能源 (能源), R² = 0.0743

  - R²差异的解释：
  - 五粮液的R² = 0.4989，说明其收益变动的49.9%可由市场收益解释
  - 中煤能源的R² = 0.0743，说明其收益变动的7.4%可由市场收益解释
  - 五粮液的R²最高（约0.50），是因为它作为白酒龙头和沪深300核心成分股，与宏观经济、消费升级及机构资金流向高度相关，其股价波动中约一半可由市场整体走势解释；而中煤能源的R²极低（约0.07），则是因为煤炭股主要受煤价、供需政策、安全生产等行业特有因素驱动，与沪深300所代表的大盘蓝筹行情关联很弱，因此市场波动只能解释其股价变动的极小部分。   
     


## 结论
- 本报告选取了覆盖银行、汽车、房地产、白酒、能源、通讯和物流七大行业的10只A股龙头，基于其2020年至2026年3月的数据，结合沪深300指数、CPI及M2等宏观指标，进行了系统的描述性统计、相关性分析和CAPM模型检验。主要结论如下：

1. 行业分化显著，白酒与能源板块表现迥异

- 防御性与进攻性并存： 白酒板块（贵州茅台、五粮液）和银行板块（招商银行）表现稳健，其中茅台实现了正的年化收益（6.66%），体现了其强大的品牌护城河和防御属性。而能源板块（中国石油、中煤能源）表现出高收益、高波动的进攻性特征，中煤能源年化收益率高达24.16%，但波动率也相应较高。

- 房地产板块深度调整： 万科A和保利地产在统计区间内经历了显著下跌，年化收益率分别为-31.52%和-12.69%，最大回撤分别高达-86.1%和-65.8%，直观地反映了房地产行业在此期间面临的严峻挑战和深度调整。

2. 行业因素是股价联动的核心驱动力

- 收益率相关系数热力图显示，同行业内部平均相关性（白酒0.83，房地产0.82）远高于跨行业平均相关性（0.29）。这量化地证明了行业层面的共同因素（如政策、供需格局、产业周期）是解释个股价格同步运动的最关键变量，对于构建多元化投资组合以分散风险具有重要指导意义。

3. 通胀（CPI）与股市关联较弱

- CPI与沪深300月度收益率的散点图显示两者呈微弱的负相关（相关系数-0.0777，p值不显著），表明通胀并非解释期间股市波动的核心因素。


4. CAPM模型有效解释了市场风险，但个股特质风险差异巨大

- Beta系数符合行业直觉： 五粮液（β=1.30）和万科A（β=1.00）的Beta值大于或等于1，表现为“进攻型”资产，其波动性高于市场。这与白酒作为顺周期消费龙头、房地产作为强周期行业的特征相吻合。而能源股（中国石油β=0.47）Beta值远小于1，体现了其与大盘蓝筹行情关联度低的特性。

- Alpha揭示超额收益来源： 在观察期内，仅万科A的Alpha值在统计上显著为负（α=-0.0013, p=0.0097），表明其收益率显著低于CAPM模型基于其承担的市场风险所给出的预期，是房地产行业整体下行压力的微观体现。其余9只股票的Alpha均不显著，说明其收益主要被市场风险所解释，没有获得显著的超额收益或损失。

- R²揭示市场解释力： 五粮液的R²最高（0.50），表明其股价波动中约有一半可由市场整体走势解释，是典型的核心蓝筹股。而中煤能源的R²极低（0.07），说明其股价主要由煤炭价格、安全生产政策等行业特有因素驱动，与大盘走势几乎无关，投资此类资产需要关注行业自身的逻辑。

**总结：**
- 在2020年至2026年的观察期内，A股市场呈现出显著的行业分化特征。以白酒、银行为代表的消费和金融板块展现了较强的稳健性，而以房地产为代表的板块则经历了深度的价值重构。CAPM模型验证了市场风险（Beta）是定价的核心因子，但个股收益率与市场整体走势的拟合度（R²）因行业属性（周期/防御）和公司特质而存在巨大差异。宏观层面，相较于通胀（CPI），以M2增速为代表的流动性环境是影响A股市场走势的更关键变量。对于投资者而言，理解个股所处的行业周期、其相对于市场的波动弹性（Beta）以及驱动其股价的特有因素，是实现风险控制和获取超额收益的基础。

